# Discovery + Silver: `crm.activities`

Ultima tabla del proyecto. `contact_id` y `opportunity_id` son FKs **opcionales** (una actividad puede no estar ligada a ninguno, a uno, o a ambos) -- ya sabiamos por `docs/calidad_datos.md` que estan vacios ~30% y ~50% de las veces respectivamente. Se mantienen nullable, no se descartan filas.

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.crm__activities", engine)
df.shape

(20000, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

activity_id               object
type                      object
subject                   object
occurred_at               object
contact_id                object
opportunity_id            object
_source_file              object
_ingested_at      datetime64[ns]
_dag_run_id               object
dtype: object


,activity_id,type,subject,occurred_at,contact_id,opportunity_id,_source_file,_ingested_at,_dag_run_id
0,ACT-00000001,note,Activity 00000001,2025-09-02 09:59:39,None,OPP-0001007,crm/activities.csv,2026-07-17 15:16:33.298675,manual__2026-07-17T15:16:30+00:00
1,ACT-00000002,email,Activity 00000002,2025-09-22 22:05:00,None,None,crm/activities.csv,2026-07-17 15:16:33.298675,manual__2026-07-17T15:16:30+00:00
2,ACT-00000003,call,Activity 00000003,2025-07-28 12:54:14,CON-0009655,OPP-0001066,crm/activities.csv,2026-07-17 15:16:33.298675,manual__2026-07-17T15:16:30+00:00
3,ACT-00000004,call,Activity 00000004,2023-05-31 13:55:19,CON-0012508,OPP-0002889,crm/activities.csv,2026-07-17 15:16:33.298675,manual__2026-07-17T15:16:30+00:00
4,ACT-00000005,email,Activity 00000005,2023-12-13 04:06:56,CON-0005585,OPP-0002958,crm/activities.csv,2026-07-17 15:16:33.298675,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados e integridad referencial (FKs opcionales)

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("activity_id duplicados:", df["activity_id"].duplicated().sum())
print()

contacts = pd.read_sql("SELECT contact_id FROM silver.crm__contacts", engine)
opportunities = pd.read_sql("SELECT opportunity_id FROM silver.crm__opportunities", engine)

con_contact = df[df["contact_id"].notna()]
con_opp = df[df["opportunity_id"].notna()]
print("contact_id huerfanos (entre los no nulos):", (~con_contact["contact_id"].isin(contacts["contact_id"])).sum())
print("opportunity_id huerfanos (entre los no nulos):", (~con_opp["opportunity_id"].isin(opportunities["opportunity_id"])).sum())
print()
print("Filas sin contact_id NI opportunity_id:", (df["contact_id"].isna() & df["opportunity_id"].isna()).sum())

Nulos por columna:
activity_id          0
type                 0
subject              0
occurred_at          0
contact_id        5976
opportunity_id    9985
_source_file         0
_ingested_at         0
_dag_run_id          0
dtype: int64

activity_id duplicados: 0

contact_id huerfanos (entre los no nulos): 0
opportunity_id huerfanos (entre los no nulos): 0

Filas sin contact_id NI opportunity_id: 2981


## 3. `type`: valores

In [4]:
print("type:")
print(df["type"].value_counts())

type:
type
email      6904
call       6093
meeting    3016
note       2028
demo       1959
Name: count, dtype: int64


## 4. Conclusion

Tabla limpia (sin nulos en columnas obligatorias, sin duplicados, 0 FKs huerfanas entre las no nulas). Las actividades sin `contact_id` ni `opportunity_id` son legitimas (actividad general, no ligada a un caso puntual) -- se mantienen.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["activity_id", "type", "subject", "occurred_at", "contact_id", "opportunity_id"]].copy()

df_silver["type"] = df_silver["type"].str.strip().str.lower()
df_silver["subject"] = df_silver["subject"].str.strip()
df_silver["occurred_at"] = pd.to_datetime(df_silver["occurred_at"])
# contact_id / opportunity_id se dejan tal cual (ya vienen como NaN de pandas para los vacios)

df_silver.head()

,activity_id,type,subject,occurred_at,contact_id,opportunity_id
0,ACT-00000001,note,Activity 00000001,2025-09-02 09:59:39,None,OPP-0001007
1,ACT-00000002,email,Activity 00000002,2025-09-22 22:05:00,None,None
2,ACT-00000003,call,Activity 00000003,2025-07-28 12:54:14,CON-0009655,OPP-0001066
3,ACT-00000004,call,Activity 00000004,2023-05-31 13:55:19,CON-0012508,OPP-0002889
4,ACT-00000005,email,Activity 00000005,2023-12-13 04:06:56,CON-0005585,OPP-0002958


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["activity_id"].is_unique
assert df_silver["contact_id"].dropna().isin(contacts["contact_id"]).all()
assert df_silver["opportunity_id"].dropna().isin(opportunities["opportunity_id"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 20000 filas listas para silver


## 7. Escribir en `silver.crm__activities`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "crm__activities",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=3000,
)
print("Escrito en silver.crm__activities")

Escrito en silver.crm__activities


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.crm__activities LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(contact_id) AS con_contact, count(opportunity_id) AS con_opp FROM silver.crm__activities", engine))
check

   filas  con_contact  con_opp
0  20000        14024    10015


,activity_id,type,subject,occurred_at,contact_id,opportunity_id,_silver_loaded_at
0,ACT-00000001,note,Activity 00000001,2025-09-02 09:59:39,None,OPP-0001007,2026-07-17 15:17:47.636604+00:00
1,ACT-00000002,email,Activity 00000002,2025-09-22 22:05:00,None,None,2026-07-17 15:17:47.636604+00:00
2,ACT-00000003,call,Activity 00000003,2025-07-28 12:54:14,CON-0009655,OPP-0001066,2026-07-17 15:17:47.636604+00:00
3,ACT-00000004,call,Activity 00000004,2023-05-31 13:55:19,CON-0012508,OPP-0002889,2026-07-17 15:17:47.636604+00:00
4,ACT-00000005,email,Activity 00000005,2023-12-13 04:06:56,CON-0005585,OPP-0002958,2026-07-17 15:17:47.636604+00:00
